# Benchmark Veri Dogrulama
yfinance baglantisi, sembol verileri, gram altin/gumus donusumu ve normalizasyon testleri.
Her hucre bagimsiz calisir. PASS = OK, AssertionError veya hata = sorun var.

In [1]:
# Hucre 1 - Kurulum ve lib import
import subprocess, sys, os

REQUIRED = ["yfinance", "plotly", "ipywidgets"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

LIB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from benchmark_engine import (
    normalize_to_100, build_benchmark_series, build_deposit_series,
    TROY_OZ_TO_GRAM, GRAM_SYMBOLS
)
from data_loader import fetch_prices, load_cpi_series, load_tcmb_rates
from chart_builder import build_performance_line_chart

print("Kurulum tamam")
print(f"TROY_OZ_TO_GRAM = {TROY_OZ_TO_GRAM}")
print(f"GRAM_SYMBOLS    = {GRAM_SYMBOLS}")

Kurulum tamam
TROY_OZ_TO_GRAM = 31.1035
GRAM_SYMBOLS    = {'SI=F', 'GC=F'}


In [2]:
# Hucre 2 - Ham fiyat testi
# yfinance'ten son 5 gunluk veri cek, her sembol icin son fiyati yazdir
SEMBOLLER = ["GC=F", "SI=F", "USDTRY=X", "EURTRY=X", "XU100.IS"]

print("yfinance'ten veri cekiliyor...")
raw = yf.download(SEMBOLLER, period="5d", auto_adjust=True, progress=False)

if isinstance(raw.columns, pd.MultiIndex):
    raw = raw["Close"]

print()
print("=" * 45)
print(f"{'Sembol':<15} {'Son Fiyat':>12} {'Tarih':>12}")
print("=" * 45)

hata_var = False
for sym in SEMBOLLER:
    if sym in raw.columns:
        seri = raw[sym].dropna()
        if len(seri) > 0:
            son_fiyat = seri.iloc[-1]
            son_tarih = seri.index[-1].strftime("%Y-%m-%d")
            print(f"{sym:<15} {son_fiyat:>12.4f} {son_tarih:>12}")
        else:
            print(f"{sym:<15} {'VERİ YOK':>12} {'---':>12}  <-- KONTROL ET")
            hata_var = True
    else:
        print(f"{sym:<15} {'SUTUN YOK':>12} {'---':>12}  <-- KONTROL ET")
        hata_var = True

print("=" * 45)
if hata_var:
    print("UYARI: Bazi semboller veri getirmedi. Internet baglantisinizi kontrol edin.")
else:
    print("PASS: Tum semboller veri getirdi")

yfinance'ten veri cekiliyor...

Sembol             Son Fiyat        Tarih
GC=F               4707.0000   2026-05-10
SI=F                 80.7150   2026-05-10
USDTRY=X             45.3318   2026-05-11
EURTRY=X             53.3637   2026-05-11
XU100.IS          15062.7002   2026-05-08
PASS: Tum semboller veri getirdi


In [3]:
# Hucre 3 - Gram altin/gumus donusum testi
# GC=F: USD/troy oz  x  USDTRY  /  31.1035  =  TL/gram
gc_usd  = raw["GC=F"].dropna().iloc[-1]
si_usd  = raw["SI=F"].dropna().iloc[-1]
usdtry  = raw["USDTRY=X"].dropna().iloc[-1]

gram_altin_tl = gc_usd * usdtry / TROY_OZ_TO_GRAM
gram_gumus_tl = si_usd * usdtry / TROY_OZ_TO_GRAM

print("=" * 50)
print(f"GC=F (USD/troy oz)      : {gc_usd:.2f}")
print(f"SI=F (USD/troy oz)      : {si_usd:.4f}")
print(f"USDTRY                  : {usdtry:.4f}")
print(f"TROY_OZ_TO_GRAM         : {TROY_OZ_TO_GRAM}")
print("=" * 50)
print(f"Gram Altin (TL/gram)    : {gram_altin_tl:.2f} TL")
print(f"Gram Gumus (TL/gram)    : {gram_gumus_tl:.4f} TL")
print("=" * 50)

# Sanity check: mantikli araliklar mi?
assert 1500 < gram_altin_tl < 15000, (
    f"Gram altin fiyati beklenmedik: {gram_altin_tl:.2f} TL "
    f"(beklenen: 1500-15000 TL)"
)
assert 10 < gram_gumus_tl < 1000, (
    f"Gram gumus fiyati beklenmedik: {gram_gumus_tl:.4f} TL "
    f"(beklenen: 10-1000 TL)"
)
print("PASS: Gram donusum sanity check")

GC=F (USD/troy oz)      : 4707.00
SI=F (USD/troy oz)      : 80.7150
USDTRY                  : 45.3318
TROY_OZ_TO_GRAM         : 31.1035
Gram Altin (TL/gram)    : 6860.22 TL
Gram Gumus (TL/gram)    : 117.6381 TL
PASS: Gram donusum sanity check


In [4]:
# Hucre 4 - Normalizasyon assert testleri

# Test 1: temel normalizasyon
test_s = pd.Series(
    [1800.0, 2700.0, 3600.0],
    index=pd.date_range("2023-01-02", periods=3, freq="B")
)
result = normalize_to_100(test_s, "2023-01-02")
assert result.iloc[0] == 100.0, f"Ilk deger 100 olmali: {result.iloc[0]}"
assert abs(result.iloc[1] - 150.0) < 1e-9
assert abs(result.iloc[2] - 200.0) < 1e-9
print("PASS: normalize_to_100 temel test")

# Test 2: start_date indexte yok - nearest forward
test_s2 = pd.Series(
    [500.0, 1000.0],
    index=pd.to_datetime(["2023-01-04", "2023-01-05"])
)
result2 = normalize_to_100(test_s2, start_date="2023-01-02")  # hafta sonu
assert result2.iloc[0] == 100.0
assert abs(result2.iloc[1] - 200.0) < 1e-9
print("PASS: normalize_to_100 nearest-forward (hafta sonu basi)")

# Test 3: gram donusum benchmark_engine ile entegrasyon
idx = pd.date_range("2023-01-02", periods=3, freq="B")
test_prices = pd.DataFrame({"GC=F": [1800.0, 1900.0, 2000.0]}, index=idx)
test_fx     = pd.Series([27.0, 27.0, 27.0], index=idx)

bench = build_benchmark_series(
    symbols=["GC=F"],
    start_date="2023-01-02",
    end_date="2023-01-06",
    prices=test_prices,
    fx_usdtry=test_fx,
    currency="TL",
)
# Baslangic=100 olmali
assert abs(bench["GC=F"].iloc[0] - 100.0) < 1e-9
# Fiyat 1800 -> 2000: +11.11%, 100 -> 111.11
expected_last = 2000 / 1800 * 100
assert abs(bench["GC=F"].iloc[2] - expected_last) < 0.001
print(f"PASS: Gram altin benchmark normalizasyonu (son deger: {bench['GC=F'].iloc[2]:.2f}, beklenen: {expected_last:.2f})")

print()
print("Tum normalizasyon testleri GECTI")

PASS: normalize_to_100 temel test
PASS: normalize_to_100 nearest-forward (hafta sonu basi)
PASS: Gram altin benchmark normalizasyonu (son deger: 111.11, beklenen: 111.11)

Tum normalizasyon testleri GECTI


In [5]:
# Hucre 5 - Son 30 gunluk canli benchmark grafigi
DRIVE_BASE = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data") + os.sep
CACHE_PATH = os.path.join(DRIVE_BASE, "cache")
os.makedirs(CACHE_PATH, exist_ok=True)

end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=45)).strftime("%Y-%m-%d")  # iş günü buffer

SYMBOLS = {
    "Gram Altin":  "GC=F",
    "Gram Gumus":  "SI=F",
    "DOLAR":       "USDTRY=X",
    "EURO":        "EURTRY=X",
    "BIST100":     "XU100.IS",
}

all_symbols = list(SYMBOLS.values()) + ["USDTRY=X"]
prices = fetch_prices(all_symbols, start=start_date, end=end_date, cache_path=CACHE_PATH)
fx_usdtry = prices["USDTRY=X"].dropna()

# Gercek 30 gun baslangicini hesapla
biz_days = prices["USDTRY=X"].dropna().index
actual_start = biz_days[-min(30, len(biz_days))].strftime("%Y-%m-%d")

benchmark_df = build_benchmark_series(
    symbols=list(SYMBOLS.values()),
    start_date=actual_start,
    end_date=end_date,
    prices=prices,
    fx_usdtry=fx_usdtry,
    currency="TL",
)

# Getiri tablosu
son = benchmark_df.dropna(how="all").iloc[-1].dropna()
sym_to_name = {v: k for k, v in SYMBOLS.items()}
print("\nSon 30 Gun Getiri Ozeti (TL, baz=100)")
print("=" * 38)
for sym, val in son.sort_values(ascending=False).items():
    isim = sym_to_name.get(sym, sym)
    isaretli = f"+{val-100:.2f}%" if val >= 100 else f"{val-100:.2f}%"
    print(f"  {isim:<14}: {val:6.2f}  ({isaretli})")
print("=" * 38)

# Grafik
fig = build_performance_line_chart(
    portfolio_series=None,
    benchmark_series=benchmark_df,
    currency_label="TL (Nominal)",
    title=f"Son 30 Is Gunu Benchmark ({actual_start} - {end_date})",
)
display(fig)


Son 30 Gun Getiri Ozeti (TL, baz=100)
  BIST100       : 117.76  (+17.76%)
  Gram Gumus    : 109.45  (+9.45%)
  EURO          : 104.12  (+4.12%)
  Gram Altin    : 103.28  (+3.28%)
  DOLAR         : 101.69  (+1.69%)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': {'bdata': ('AAAAAAAAAABg4VLYRP0GQAA8gN3nrr' ... 'l+7VqzBkDgG6KeFzsKQOAbop4XOwpA'),
                             'dtype': 'f8'},
              'hovertemplate': ('<b>GC=F</b><br>Tarih: %{x|%d.%' ... 'ustomdata:.1f}%<extra></extra>'),
              'line': {'color': '#F4A460', 'dash': 'dash', 'width': 1},
              'name': 'GC=F',
              'opacity': 0.6,
              'type': 'scatter',
              'x': array(['2026-03-31T00:00:00.000000', '2026-04-01T00:00:00.000000',
                          '2026-04-02T00:00:00.000000', '2026-04-03T00:00:00.000000',
                          '2026-04-06T00:00:00.000000', '2026-04-07T00:00:00.000000',
                          '2026-04-08T00:00:00.000000', '2026-04-09T00:00:00.000000',
                          '2026-04-10T00:00:00.000000', '2026-04-13T00:00:00.000000',
                          '2026-04-14T00:00:00.000000', '2026-04-15T00:00:00.000000',
                          '2026-04-16T00:00:00.000000', '2026-04-17T00:00:00.000000',
                          '2026-04-20T00:00:00.000000', '2026-04-21T00:00:00.000000',
                          '2026-04-22T00:00:00.000000', '2026-04-23T00:00:00.000000',
                          '2026-04-24T00:00:00.000000', '2026-04-27T00:00:00.000000',
                          '2026-04-28T00:00:00.000000', '2026-04-29T00:00:00.000000',
                          '2026-04-30T00:00:00.000000', '2026-05-01T00:00:00.000000',
                          '2026-05-04T00:00:00.000000', '2026-05-05T00:00:00.000000',
                          '2026-05-06T00:00:00.000000', '2026-05-07T00:00:00.000000',
                          '2026-05-08T00:00:00.000000', '2026-05-11T00:00:00.000000'],
                         dtype='datetime64[us]'),
              'y': {'bdata': ('AAAAAAAAWUALl8Im6rdZQA9g97krB1' ... 'Rr15q1WUDfEPW82NFZQN8Q9bzY0VlA'),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAAAAAAAAACApzbf4YT4PwCEjYiOuA' ... 'tz3k4JIUBIdQjB9+ciQEh1CMH35yJA'),
                             'dtype': 'f8'},
              'hovertemplate': ('<b>SI=F</b><br>Tarih: %{x|%d.%' ... 'ustomdata:.1f}%<extra></extra>'),
              'line': {'color': '#A9A9A9', 'dash': 'dash', 'width': 1},
              'name': 'SI=F',
              'opacity': 0.6,
              'type': 'scatter',
              'x': array(['2026-03-31T00:00:00.000000', '2026-04-01T00:00:00.000000',
                          '2026-04-02T00:00:00.000000', '2026-04-03T00:00:00.000000',
                          '2026-04-06T00:00:00.000000', '2026-04-07T00:00:00.000000',
                          '2026-04-08T00:00:00.000000', '2026-04-09T00:00:00.000000',
                          '2026-04-10T00:00:00.000000', '2026-04-13T00:00:00.000000',
                          '2026-04-14T00:00:00.000000', '2026-04-15T00:00:00.000000',
                          '2026-04-16T00:00:00.000000', '2026-04-17T00:00:00.000000',
                          '2026-04-20T00:00:00.000000', '2026-04-21T00:00:00.000000',
                          '2026-04-22T00:00:00.000000', '2026-04-23T00:00:00.000000',
                          '2026-04-24T00:00:00.000000', '2026-04-27T00:00:00.000000',
                          '2026-04-28T00:00:00.000000', '2026-04-29T00:00:00.000000',
                          '2026-04-30T00:00:00.000000', '2026-05-01T00:00:00.000000',
                          '2026-05-04T00:00:00.000000', '2026-05-05T00:00:00.000000',
                          '2026-05-06T00:00:00.000000', '2026-05-07T00:00:00.000000',
                          '2026-05-08T00:00:00.000000', '2026-05-11T00:00:00.000000'],
                         dtype='datetime64[us]'),
              'y': {'bdata': ('AAAAAAAAWUCe2nyHE2JZQOCTu4s7Wl' ... 'HO2ykhW0CpDiH4/lxbQKkOIfj+XFtA'),
                    'dtype': 'f8'}},
             {'customdata': {'bdata': ('AAAAAAAAAAAA0L/m6d+lvwCgNNxdyJ' ... 'UG0EYg+z+ALkRMDfr6P4AuREwN+vo/'),
                             'dtype': 'f8'},
              'hovertempl